# Acoustic-unit clustering and K-study

In [ ]:

from pathlib import Path
import os
import json
import math
import logging
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm

from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler


In [ ]:

cwd = Path.cwd().resolve()

candidate_roots = [cwd, cwd.parent]
PROJECT_ROOT = None
for cand in candidate_roots:
    if (cand / "configs" / "config.yaml").exists():
        PROJECT_ROOT = cand
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find configs/config.yaml from the current notebook location.")

PROJECT_ROOT


In [ ]:

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

defaults = {
    "paths": {
        "output_dir": "output",
    },
    "ssl": {
        "output_subdir": "01_ssl_frames",
    },
    "segmentation": {
        "output_subdir": "02_segments",
    },
    "clustering": {
        "segment_vectors_subdir": "03_segment_vectors",
        "kstudy_subdir": "04_kstudy",
        "k_values": [32, 48, 64, 80, 96],
        "random_seeds": [7, 19, 31, 43, 55],
        "batch_size": 256,
        "max_iter": 300,
        "n_init": 10,
        "use_pca": True,
        "pca_components": 128,
        "silhouette_max_samples": 3000,
        "final_model_seed": 31,
        "overwrite_segment_vectors": False,
        "overwrite_kstudy": False,
    }
}

def deep_merge(a, b):
    out = dict(a)
    for k, v in b.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_merge(out[k], v)
        else:
            out[k] = v
    return out

config = deep_merge(defaults, config or {})
config


In [ ]:

OUTPUT_DIR = PROJECT_ROOT / config["paths"]["output_dir"]
SSL_DIR = OUTPUT_DIR / config["ssl"]["output_subdir"]
SEG_DIR = OUTPUT_DIR / config["segmentation"]["output_subdir"]
SEGMENT_VECTOR_DIR = OUTPUT_DIR / config["clustering"]["segment_vectors_subdir"]
KSTUDY_DIR = OUTPUT_DIR / config["clustering"]["kstudy_subdir"]
LOG_DIR = OUTPUT_DIR / "logs"

for p in [SEGMENT_VECTOR_DIR, KSTUDY_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEGMENTS_CSV = SEG_DIR / "segments.csv"
SEGMENT_SUMMARY_CSV = SEG_DIR / "segment_summary.csv"
FRAME_INDEX_CSV = SSL_DIR / "frame_index.csv"
SEGMENT_VECTORS_NPZ = SEGMENT_VECTOR_DIR / "segment_vectors.npz"
KSTUDY_RESULTS_CSV = KSTUDY_DIR / "kstudy_results.csv"
PROVISIONAL_MD = KSTUDY_DIR / "provisional_k_selection.md"
MODEL_PKL = KSTUDY_DIR / "kmeans_model.pkl"
LOG_FILE = LOG_DIR / "03_cluster_segments_kstudy.log"

logger = logging.getLogger("03_cluster_segments_kstudy")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(fh)

logger.info("Project root: %s", PROJECT_ROOT)
logger.info("Starting 03_cluster_segments_kstudy notebook")
print("Log file:", LOG_FILE)


In [ ]:

segments_df = pd.read_csv(SEGMENTS_CSV)
segment_summary_df = pd.read_csv(SEGMENT_SUMMARY_CSV)
frame_index_df = pd.read_csv(FRAME_INDEX_CSV)

display(segments_df.head())
display(segment_summary_df.head())
display(frame_index_df.head())

print("segments_df shape:", segments_df.shape)
print("segment_summary_df shape:", segment_summary_df.shape)
print("frame_index_df shape:", frame_index_df.shape)

required_segment_cols = [
    "clip_id", "segment_id", "segment_index", "frame_file",
    "start_frame", "end_frame_exclusive", "num_frames",
    "start_sec", "end_sec", "duration_sec", "clip_duration_sec"
]
missing_segment_cols = [c for c in required_segment_cols if c not in segments_df.columns]
if missing_segment_cols:
    raise ValueError(f"segments.csv is missing required columns: {missing_segment_cols}")

required_frame_cols = ["clip_id", "frame_file", "num_frames", "hidden_size"]
missing_frame_cols = [c for c in required_frame_cols if c not in frame_index_df.columns]
if missing_frame_cols:
    raise ValueError(f"frame_index.csv is missing required columns: {missing_frame_cols}")


In [ ]:

# Quick integrity checks before building vectors.
coverage = (
    segments_df.groupby("clip_id")
    .agg(
        segment_count=("segment_id", "size"),
        min_start=("start_sec", "min"),
        max_end=("end_sec", "max"),
        clip_duration=("clip_duration_sec", "first"),
        total_segment_dur=("duration_sec", "sum"),
    )
    .reset_index()
)
coverage["end_gap_sec"] = coverage["clip_duration"] - coverage["max_end"]
coverage["coverage_gap_sec"] = coverage["clip_duration"] - coverage["total_segment_dur"]

display(coverage.describe(include="all"))
print("Any negative duration segments?", bool((segments_df["duration_sec"] < 0).any()))
print("Any zero-frame segments?", bool((segments_df["num_frames"] <= 0).any()))
print("Max |end gap|:", float(coverage["end_gap_sec"].abs().max()))
print("Max |coverage gap|:", float(coverage["coverage_gap_sec"].abs().max()))

logger.info("Loaded %d segments across %d clips", len(segments_df), segments_df['clip_id'].nunique())


In [ ]:

def load_frame_array(frame_path: Path):
    arr = np.load(frame_path, allow_pickle=False)
    key = "frames"
    if key not in arr.files:
        # fallback if the notebook saved a different key name
        candidate = [k for k in arr.files if arr[k].ndim == 2]
        if not candidate:
            raise KeyError(f"No 2D frame array found in {frame_path}")
        key = candidate[0]
    frames = arr[key]
    frame_times = arr["frame_times_sec"] if "frame_times_sec" in arr.files else None
    return frames, frame_times

def aggregate_segment_vector(frames_2d: np.ndarray) -> np.ndarray:
    # Segment vector = mean SSL embedding over the segment.
    return frames_2d.mean(axis=0).astype(np.float32)

overwrite_segment_vectors = bool(config["clustering"]["overwrite_segment_vectors"])


In [ ]:

if SEGMENT_VECTORS_NPZ.exists() and not overwrite_segment_vectors:
    print("Loading existing segment_vectors.npz")
    logger.info("Loading existing segment_vectors.npz")
    sv = np.load(SEGMENT_VECTORS_NPZ, allow_pickle=False)
    vector_matrix = sv["vectors"]
    vector_segment_ids = sv["segment_ids"].astype(str)
    vector_clip_ids = sv["clip_ids"].astype(str)
    vector_segment_index = sv["segment_index"]
    vector_duration_sec = sv["duration_sec"]
    vector_start_sec = sv["start_sec"]
    vector_end_sec = sv["end_sec"]
else:
    rows = []
    frame_cache = {}

    for row in tqdm(segments_df.itertuples(index=False), total=len(segments_df), desc="Building segment vectors"):
        frame_path = Path(row.frame_file)
        if not frame_path.is_absolute():
            frame_path = (PROJECT_ROOT / frame_path).resolve()
        if not frame_path.exists():
            # also allow relative-to-output path
            alt = (PROJECT_ROOT / row.frame_file.replace("\\", "/")).resolve()
            if alt.exists():
                frame_path = alt
        if frame_path not in frame_cache:
            frames_2d, _ = load_frame_array(frame_path)
            frame_cache[frame_path] = frames_2d
        else:
            frames_2d = frame_cache[frame_path]

        start_i = int(row.start_frame)
        end_i = int(row.end_frame_exclusive)
        if start_i < 0 or end_i > len(frames_2d) or end_i <= start_i:
            raise ValueError(
                f"Invalid frame slice for {row.segment_id}: start={start_i}, end={end_i}, total={len(frames_2d)}"
            )
        seg_frames = frames_2d[start_i:end_i]
        vec = aggregate_segment_vector(seg_frames)
        rows.append(
            (
                row.segment_id,
                row.clip_id,
                int(row.segment_index),
                float(row.start_sec),
                float(row.end_sec),
                float(row.duration_sec),
                vec,
            )
        )

    vector_segment_ids = np.array([r[0] for r in rows], dtype="U256")
    vector_clip_ids = np.array([r[1] for r in rows], dtype="U256")
    vector_segment_index = np.array([r[2] for r in rows], dtype=np.int32)
    vector_start_sec = np.array([r[3] for r in rows], dtype=np.float32)
    vector_end_sec = np.array([r[4] for r in rows], dtype=np.float32)
    vector_duration_sec = np.array([r[5] for r in rows], dtype=np.float32)
    vector_matrix = np.stack([r[6] for r in rows]).astype(np.float32)

    np.savez_compressed(
        SEGMENT_VECTORS_NPZ,
        vectors=vector_matrix,
        segment_ids=vector_segment_ids,
        clip_ids=vector_clip_ids,
        segment_index=vector_segment_index,
        start_sec=vector_start_sec,
        end_sec=vector_end_sec,
        duration_sec=vector_duration_sec,
        created_at_utc=np.array([datetime.now(timezone.utc).isoformat()]),
    )
    logger.info("Saved segment vectors to %s", SEGMENT_VECTORS_NPZ)

print("vector_matrix shape:", vector_matrix.shape)
print("dtype:", vector_matrix.dtype)
print("Unique clips represented:", len(np.unique(vector_clip_ids)))


In [ ]:

segment_vector_preview = pd.DataFrame({
    "segment_id": vector_segment_ids[:5],
    "clip_id": vector_clip_ids[:5],
    "segment_index": vector_segment_index[:5],
    "start_sec": vector_start_sec[:5],
    "end_sec": vector_end_sec[:5],
    "duration_sec": vector_duration_sec[:5],
})
display(segment_vector_preview)

print("Vector L2 norm stats:")
vector_norms = np.linalg.norm(vector_matrix, axis=1)
display(pd.Series(vector_norms).describe())


In [ ]:

# Prepare the matrix for clustering.
use_pca = bool(config["clustering"]["use_pca"])
pca_components_cfg = int(config["clustering"]["pca_components"])

scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(vector_matrix)

pca = None
if use_pca:
    max_allowed = min(X_scaled.shape[0] - 1, X_scaled.shape[1], pca_components_cfg)
    if max_allowed >= 2:
        pca = PCA(n_components=max_allowed, random_state=0)
        X_cluster = pca.fit_transform(X_scaled).astype(np.float32)
        print(f"Using PCA: {X_scaled.shape[1]} -> {X_cluster.shape[1]}")
        print("Explained variance ratio sum:", float(pca.explained_variance_ratio_.sum()))
    else:
        X_cluster = X_scaled.astype(np.float32)
        print("Skipping PCA because max allowed components < 2")
else:
    X_cluster = X_scaled.astype(np.float32)
    print("PCA disabled")

display(pd.Series(np.linalg.norm(X_cluster, axis=1)).describe())
logger.info("Clustering matrix shape: %s", X_cluster.shape)


In [ ]:

k_values = list(config["clustering"]["k_values"])
seeds = list(config["clustering"]["random_seeds"])
batch_size = int(config["clustering"]["batch_size"])
max_iter = int(config["clustering"]["max_iter"])
n_init = int(config["clustering"]["n_init"])
silhouette_max_samples = int(config["clustering"]["silhouette_max_samples"])
overwrite_kstudy = bool(config["clustering"]["overwrite_kstudy"])

k_values, seeds


In [ ]:

def run_kmeans_once(X, k, seed, batch_size=256, max_iter=300, n_init=10):
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=seed,
        batch_size=batch_size,
        max_iter=max_iter,
        n_init=n_init,
        reassignment_ratio=0.01,
    )
    labels = model.fit_predict(X)
    counts = np.bincount(labels, minlength=k)
    return model, labels, counts

def safe_silhouette(X, labels, max_samples=3000, seed=0):
    n_labels = len(np.unique(labels))
    n = len(labels)
    if n_labels < 2 or n_labels >= n:
        return np.nan
    if n > max_samples:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(n, size=max_samples, replace=False))
        return float(silhouette_score(X[idx], labels[idx]))
    return float(silhouette_score(X, labels))

def safe_calinski(X, labels):
    n_labels = len(np.unique(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return np.nan
    return float(calinski_harabasz_score(X, labels))

def safe_davies(X, labels):
    n_labels = len(np.unique(labels))
    if n_labels < 2 or n_labels >= len(labels):
        return np.nan
    return float(davies_bouldin_score(X, labels))

def pairwise_stability(label_runs):
    if len(label_runs) < 2:
        return np.nan, np.nan
    aris = []
    for i in range(len(label_runs)):
        for j in range(i + 1, len(label_runs)):
            aris.append(adjusted_rand_score(label_runs[i], label_runs[j]))
    return float(np.mean(aris)), float(np.std(aris))

def cluster_size_stats(counts):
    counts = np.array(counts, dtype=np.int64)
    nonzero = counts[counts > 0]
    probs = nonzero / nonzero.sum()
    entropy = -np.sum(probs * np.log(probs + 1e-12))
    normalized_entropy = float(entropy / np.log(len(nonzero))) if len(nonzero) > 1 else 0.0
    return {
        "min_cluster_size": int(nonzero.min()) if len(nonzero) else 0,
        "median_cluster_size": float(np.median(nonzero)) if len(nonzero) else 0.0,
        "max_cluster_size": int(nonzero.max()) if len(nonzero) else 0,
        "normalized_cluster_entropy": normalized_entropy,
        "singleton_clusters": int(np.sum(nonzero == 1)),
    }

def choose_provisional_k(results_df):
    df = results_df.copy()
    # Normalize metrics to [0, 1] so we can combine them conservatively.
    def norm_col(col, higher_is_better=True):
        s = df[col].astype(float)
        if s.isna().all():
            return pd.Series(np.nan, index=df.index)
        lo, hi = s.min(skipna=True), s.max(skipna=True)
        if pd.isna(lo) or pd.isna(hi) or hi == lo:
            base = pd.Series(1.0, index=df.index)
        else:
            base = (s - lo) / (hi - lo)
        return base if higher_is_better else 1.0 - base

    score = (
        0.40 * norm_col("mean_pairwise_ari", higher_is_better=True).fillna(0.0) +
        0.20 * norm_col("mean_silhouette", higher_is_better=True).fillna(0.0) +
        0.15 * norm_col("mean_calinski_harabasz", higher_is_better=True).fillna(0.0) +
        0.10 * norm_col("mean_davies_bouldin", higher_is_better=False).fillna(0.0) +
        0.10 * norm_col("mean_normalized_cluster_entropy", higher_is_better=True).fillna(0.0) +
        0.05 * norm_col("mean_min_cluster_size", higher_is_better=True).fillna(0.0)
    )

    # Mild penalty for too many singleton clusters.
    penalty = np.where(df["mean_singleton_clusters"].fillna(0) > 0, 0.05, 0.0)
    df["selection_score"] = score - penalty
    best_idx = df["selection_score"].idxmax()
    return df, df.loc[best_idx]


In [ ]:

if KSTUDY_RESULTS_CSV.exists() and not overwrite_kstudy:
    print("Loading existing kstudy_results.csv")
    kstudy_results_df = pd.read_csv(KSTUDY_RESULTS_CSV)
else:
    run_rows = []
    summary_rows = []

    for k in tqdm(k_values, desc="K-study"):
        labels_by_seed = []
        per_seed_rows = []

        for seed in seeds:
            model, labels, counts = run_kmeans_once(
                X_cluster, k=k, seed=seed,
                batch_size=batch_size,
                max_iter=max_iter,
                n_init=n_init,
            )
            labels_by_seed.append(labels)

            size_info = cluster_size_stats(counts)
            row = {
                "k": int(k),
                "seed": int(seed),
                "inertia": float(model.inertia_),
                "silhouette": safe_silhouette(X_cluster, labels, max_samples=silhouette_max_samples, seed=seed),
                "calinski_harabasz": safe_calinski(X_cluster, labels),
                "davies_bouldin": safe_davies(X_cluster, labels),
                **size_info,
            }
            per_seed_rows.append(row)
            run_rows.append(row)

        mean_ari, std_ari = pairwise_stability(labels_by_seed)
        seed_df = pd.DataFrame(per_seed_rows)

        summary_rows.append({
            "k": int(k),
            "n_segments": int(X_cluster.shape[0]),
            "n_seeds": int(len(seeds)),
            "mean_pairwise_ari": mean_ari,
            "std_pairwise_ari": std_ari,
            "mean_inertia": float(seed_df["inertia"].mean()),
            "std_inertia": float(seed_df["inertia"].std(ddof=0)),
            "mean_silhouette": float(seed_df["silhouette"].mean()),
            "std_silhouette": float(seed_df["silhouette"].std(ddof=0)),
            "mean_calinski_harabasz": float(seed_df["calinski_harabasz"].mean()),
            "std_calinski_harabasz": float(seed_df["calinski_harabasz"].std(ddof=0)),
            "mean_davies_bouldin": float(seed_df["davies_bouldin"].mean()),
            "std_davies_bouldin": float(seed_df["davies_bouldin"].std(ddof=0)),
            "mean_min_cluster_size": float(seed_df["min_cluster_size"].mean()),
            "mean_median_cluster_size": float(seed_df["median_cluster_size"].mean()),
            "mean_max_cluster_size": float(seed_df["max_cluster_size"].mean()),
            "mean_normalized_cluster_entropy": float(seed_df["normalized_cluster_entropy"].mean()),
            "mean_singleton_clusters": float(seed_df["singleton_clusters"].mean()),
        })

    kstudy_results_df = pd.DataFrame(summary_rows).sort_values("k").reset_index(drop=True)
    kstudy_results_df.to_csv(KSTUDY_RESULTS_CSV, index=False)
    pd.DataFrame(run_rows).to_csv(KSTUDY_DIR / "kstudy_runs_by_seed.csv", index=False)
    logger.info("Saved K-study results to %s", KSTUDY_RESULTS_CSV)

display(kstudy_results_df)


In [ ]:

scored_kstudy_df, best_row = choose_provisional_k(kstudy_results_df)
display(scored_kstudy_df.sort_values("selection_score", ascending=False))

best_k = int(best_row["k"])
print("Provisional K selected:", best_k)
print(best_row.to_dict())

logger.info("Provisional K selected: %d", best_k)


In [ ]:

# Fit the final provisional model using the selected K and a fixed final seed.
final_seed = int(config["clustering"]["final_model_seed"])

final_model, final_labels, final_counts = run_kmeans_once(
    X_cluster,
    k=best_k,
    seed=final_seed,
    batch_size=batch_size,
    max_iter=max_iter,
    n_init=n_init,
)

final_size_info = cluster_size_stats(final_counts)
print("Final cluster size info:", final_size_info)
print("Unique clusters used:", len(np.unique(final_labels)))

model_bundle = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "selected_k": best_k,
    "final_seed": final_seed,
    "scaler": scaler,
    "pca": pca,
    "model": final_model,
    "segment_vectors_npz": str(SEGMENT_VECTORS_NPZ),
    "kstudy_results_csv": str(KSTUDY_RESULTS_CSV),
    "k_values_tested": k_values,
    "seeds_tested": seeds,
    "n_segments": int(X_cluster.shape[0]),
    "input_dim_raw": int(vector_matrix.shape[1]),
    "input_dim_cluster": int(X_cluster.shape[1]),
    "cluster_size_info": final_size_info,
}
joblib.dump(model_bundle, MODEL_PKL)
logger.info("Saved final provisional model bundle to %s", MODEL_PKL)

MODEL_PKL


In [ ]:

# Write the provisional selection note.
selection_lines = []
selection_lines.append("# Provisional K Selection")
selection_lines.append("")
selection_lines.append(f"- Generated at: {datetime.now(timezone.utc).isoformat()}")
selection_lines.append(f"- Number of segments clustered: {X_cluster.shape[0]}")
selection_lines.append(f"- Raw segment vector dimension: {vector_matrix.shape[1]}")
selection_lines.append(f"- Clustering input dimension: {X_cluster.shape[1]}")
selection_lines.append(f"- K values tested: {k_values}")
selection_lines.append(f"- Seeds tested: {seeds}")
selection_lines.append("")
selection_lines.append("## Selection principle")
selection_lines.append("")
selection_lines.append(
    "The provisional K was selected using unsupervised criteria only: "
    "pairwise ARI stability across random seeds, silhouette, Calinski-Harabasz, "
    "Davies-Bouldin, cluster-size balance, and a small penalty for singleton clusters."
)
selection_lines.append("")
selection_lines.append("## Selected K")
selection_lines.append("")
selection_lines.append(f"- **Provisional K = {best_k}**")
selection_lines.append(f"- Mean pairwise ARI: {best_row['mean_pairwise_ari']:.4f}")
selection_lines.append(f"- Mean silhouette: {best_row['mean_silhouette']:.4f}")
selection_lines.append(f"- Mean Calinski-Harabasz: {best_row['mean_calinski_harabasz']:.4f}")
selection_lines.append(f"- Mean Davies-Bouldin: {best_row['mean_davies_bouldin']:.4f}")
selection_lines.append(f"- Mean min cluster size: {best_row['mean_min_cluster_size']:.2f}")
selection_lines.append(f"- Mean normalized cluster entropy: {best_row['mean_normalized_cluster_entropy']:.4f}")
selection_lines.append("")
selection_lines.append("## Notes")
selection_lines.append("")
selection_lines.append(
    "This is a provisional Layer-1 choice for token inventory size. "
    "It was not selected from downstream classification accuracy."
)
selection_lines.append(
    "If segmentation is later re-tuned, this K study should be rerun because the unit inventory depends on the segment population."
)

with open(PROVISIONAL_MD, "w", encoding="utf-8") as f:
    f.write("\n".join(selection_lines))

print(PROVISIONAL_MD.read_text(encoding="utf-8"))
logger.info("Saved provisional K selection note to %s", PROVISIONAL_MD)


In [ ]:

# Helpful preview table for the next notebook.
cluster_assign_preview = pd.DataFrame({
    "segment_id": vector_segment_ids,
    "clip_id": vector_clip_ids,
    "segment_index": vector_segment_index,
    "start_sec": vector_start_sec,
    "end_sec": vector_end_sec,
    "duration_sec": vector_duration_sec,
    "provisional_cluster": final_labels,
})
display(cluster_assign_preview.head(20))

print("Per-cluster counts:")
display(pd.Series(final_labels).value_counts().sort_index().head(20))
